# CS4765/6765 Assignment 2: Sentiment Analysis for climate-related text in corporate disclosures

**Due 14 October**

We've seen the problem of sentiment analysis in class. In this
assignment you will build your own sentiment analysis system for a three-way sentiment analysis task for climate-related text in corporate disclosures. You will consider logistic regression using several sets of features, including the example features from the lecture and the textbook, count-based and tf-idf-based unigram features, and features of your choosing. You will compare these approaches to each other and a most-frequent class baseline.

Read through this notebook in its entirety before getting started. You should only make changes / write code in parts of the notebook where the instructions ask you to do so. These are indicated with TODO throughout.

Your code must be able to run on the NLP VM on the lab machines. You may only import additional modules where the instructions indicate that you may do so.


## Data

I've provided you with the following files for this assignment: `train-sample.csv`, `dev.csv`, `test.csv`.

Each of these files is a CSV file. You will use `train-sample.csv` for training models, `dev.csv` for evaluation during model development and for model refinement, and `test.csv` for final evaluation.

Each row in each file is an instance consisting of a text, and a label. (Each row also contains an id number, which we won't use in this assignment.) Each text is a climate-related paragraph from an annual report from a company. Each label is 0, 1 or 2. The labels have the following meanings:

- 2: Opportunity arising due to climate change (positive sentiment)

- 1: Neutral

- 0: Risk or threat that negatively impacts an entity (negative sentiment)

`train-sample.csv`, `dev.csv`, and `test.csv` contain 800, 200, and 300 instances, respectively.

The starter code handles reading these files for you. Do not modify these parts of starter code. You should only modify the starter code where you are instructed to (indicated with TODO).

The data for this assignment is from the [ClimateBERT climate sentiment dataset](https://huggingface.co/datasets/climatebert/climate_sentiment). The license is [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) You are of course welcome to read more about this dataset, but it is not necessary to do so to be able to do this assignment. To create the data for this assignment, I converted the original dataset files from parquet to CSV. The original dataset does not include a development set. I randomly split the training data into 80% training and 20% dev to create `train-sample.csv` and `dev.csv`. `test.csv` is the same test data as the original dataset (but formatted as CSV instead of parquet).





## Experimental Setup

Throughout this assignment, we will always use the training data (`train-sample.csv`) for training models. We will implement our models and then use the development data (`dev.csv`) for preliminary evaluation. Once we've completed this, we will do our final evaluation on the test data (`test.csv`). The starter code guides you through this process.


In [ ]:
# Helper function to load a data file
import csv

def get_texts_and_labels(fname):
    csv_reader = csv.reader(open(fname, encoding='utf8'))
    # Ignore header row
    next(csv_reader)
    texts = []
    labels = []
    for line in csv_reader:
        _,text,label = line
        label = int(label)
        texts.append(text)
        labels.append(label)
    return texts,labels

In [ ]:
# Load the training and dev data
train_texts, train_labels = get_texts_and_labels('data/train-sample.csv')
dev_texts, dev_labels = get_texts_and_labels('data/dev.csv')

# Some sanity checks
assert len(train_texts) == len(train_labels)
assert len(dev_texts) == len(dev_labels)

for label in train_labels + dev_labels:
    assert 0 <= label <= 2

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# A helper function to compute macro averaged P, R, and F1 and accuracy.
# Uses implementantions of evaluation metrics from sklearn.
def get_results(gold_labels, predicted_labels):
    p,r,f,_ = precision_recall_fscore_support(gold_labels, 
                                              predicted_labels,
                                              average='macro',
                                              zero_division=0)
    acc = accuracy_score(gold_labels, predicted_labels)
    return p,r,f,acc    

# A helper function to print out the above evaluation metrics.
def print_results(gold_labels, predicted_labels):
    p,r,f,acc = get_results(gold_labels, predicted_labels)
    
    print("Precision: ", p)
    print("Recall: ", r)
    print("F1: ", f)
    print("Accuracy: ", acc)
    print()

## Most-frequent Class Baseline

The starter code below trains a most-frequent class baseline on the training data and evaluates it on the dev data. (Note that sklearn also includes an implementation of a most-frequent class baseline, which you might find useful for your projects: https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html)

In [ ]:
# A most-frequent class baseline
class Baseline:
    def __init__(self, labels):
        self.train(labels)

    def train(self, labels):
        # Count classes to determine which is the most frequent
        label_freqs = {}
        for l in labels:
            label_freqs[l] = label_freqs.get(l, 0) + 1
        self.mfc = sorted(label_freqs, reverse=True, 
                          key=lambda x : label_freqs[x])[0]
    
    def predict(self, test_instance):
        # Ignore the test instance and always return the most frequent class
        return self.mfc


In [ ]:
baseline_classifier = Baseline(train_labels)
baseline_dev_predictions = [baseline_classifier.predict(x) for x in dev_texts]
print_results(dev_labels, baseline_dev_predictions)

## Example Features from Textbook

The code below implements the example logistic regression features from lecture and the textbook (i.e., Chapter 4, Figure 4.2).

In [ ]:
import re

# A very simple tokenizer. This implements sklearn's default
# word tokenization, which applies case folding and treats
# alphanumeric sequences of 2 or more characters as tokens.
# Punctuation is treated as a token separator, i.e.,
# punctuation never occurs in the resulting tokens.
word_tokenize_pattern = re.compile(r"(?u)\b\w\w+\b")
def word_tokenize(s, apply_case_folding=True):
    return [x.lower() for x in word_tokenize_pattern.findall(s)]


In [ ]:
# Here is an example document from the training data and the result of tokenizing it
print(train_texts[58])
print(word_tokenize(train_texts[58]))

In [ ]:
import math, nltk
import numpy as np

# First and second person pronouns based on information from 
# https://en.wikipedia.org/wiki/Personal_pronouns_in_English
first_second_pronouns = set(['i', 'me', 'my', 'mine', 'myself',
                             'we', 'us', 'our', 'ours', 'ourselves',
                             'you', 'your', 'yours', 'yourself', 'yourselves'])

# Lists of positive and negative sentiment words from the opinion lexicon included in NLTK
# https://www.nltk.org/_modules/nltk/corpus/reader/opinion_lexicon.html
# (If you've already downloaded this data you can comment the following line out.)
nltk.download('opinion_lexicon')
pos_words = set(nltk.corpus.opinion_lexicon.positive())
neg_words = set(nltk.corpus.opinion_lexicon.negative())

# Return a feature vector for the document doc
# The features correspond to the example from lecture and the textbook
# (i.e., Chapter 4, Figure 4.2)
def slp_features_for_doc(doc):
    tokens = word_tokenize(doc)
    x1 = len([t for t in tokens if t in pos_words])
    x2 = len([t for t in tokens if t in neg_words])
    x3 = 1 if "no" in tokens else 0
    x4 = 1 if "!" in doc else 0
    x5 = int(any([t in first_second_pronouns for t in tokens]))
    x6 = math.log(len(tokens))
    return np.array([x1, x2, x3, x4, x5, x6])


In [ ]:
# Here is an example document from the training data and its feature vector
print(train_texts[425])
print(slp_features_for_doc(train_texts[425]))

The code below uses scikit-learn (sklearn) to train a logistic regression classifier with the example features on the training data and evaluate it on the dev data.

Scikit-learn is a popular tool for doing many machine learning tasks in Python. It includes implementations of many classifiers (including logistic regression). Read the comments in the code below to learn how to use it.

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# slp_train_features is a Dx6 matrix, where D is the number of training documents
# and 6 is the number of features
slp_train_features = np.vstack([slp_features_for_doc(x) for x in train_texts])

# Train a logistic regression classifier on the training
# data. A wide range of options are available. sklearn uses
# L2 regularization by default. The algorithm for minimizing
# the loss during training is LBFGS instead of SGD, which we
# saw in lecture. The random_state is set to 0  (an 
# arbitrarily chosen number) to help ensure results are 
# consistent from run to run.
# http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
lr_slp_classifier = LogisticRegression(random_state=0).fit(slp_train_features, train_labels)

# slp_dev_features is a Dx6 matrix, where here D is the number of
# dev documents
slp_dev_features = np.vstack([slp_features_for_doc(x) for x in dev_texts])

# Evaluate the classifier on the dev data
lr_slp_dev_predictions = lr_slp_classifier.predict(slp_dev_features)
print_results(dev_labels, lr_slp_dev_predictions)


## Part 1: Extended features (2 marks)

In this part of the assignment you will consider extending the example features from the textbook with features of your own choosing. Based on the discussion of document classification, logistic regression, and sentiment analysis in class, or analysis of the training and development data, identify some features (e.g., 1-5 or so) that you believe could potentially give an improvement when combined with the example features. For this part of the assignment, focus on features targeting specific properties of a document, such as for the example features (as opposed to more-general features, such as counts for all unigrams in a document, which we will consider later in the assignment).

(I've included some examples of analysis of the training and dev data below that could be helpful. You are of course welcome to carry out further analyses, although you should not look at the test data yet.) 

Some ideas of features to consider could include the presence or count of specific words, punctuation marks, or sets of characters, that could be indicative of the class. However, you are by no means limited to these ideas.

Answer the questions below. Implement your new features below that. Then run the code below that to train a logistic regression classifier using these features on the training data and evaluate it on the dev data. This classifier should improve over the classifier using just the example features.


Answer the following questions:

1. Which features are you adding? Provide a brief explanation.

I add two count features. The first counts words associated with climate risk or harm (for example, *risk*, *threat*, *vulnerability*, *damage*, and *loss*). The second counts words associated with climate opportunities or benefits (for example, *opportunity*, *benefit*, *innovation*, *renewable*, *efficiency*, and *savings*).
   
2. Why might we reasonably expect that a logistic regression classifier using these features and the example features would improve over a logistic regression classifier using just the example features?

The original opinion lexicon was designed for general sentiment, whereas the labels in this dataset describe climate-related risks and opportunities. Domain-specific terms such as *exposure* or *renewable* may therefore distinguish the classes even when they are absent from, or weakly represented by, a general sentiment lexicon. Counts also retain evidence when several such terms occur in one paragraph.

In [ ]:
def new_features_for_doc(doc):
    #new_features is a list containing the new features.
    new_features = []

    risk_words = set(['risk', 'risks', 'risky', 'threat', 'threats',
                      'exposure', 'exposed', 'vulnerability', 'vulnerable',
                      'adverse', 'adversely', 'damage', 'damages', 'loss',
                      'losses', 'disruption', 'disruptions', 'hazard',
                      'hazards', 'impact', 'impacts', 'negative', 'negatively'])
    opportunity_words = set(['opportunity', 'opportunities', 'benefit',
                             'benefits', 'beneficial', 'growth', 'advantage',
                             'advantages', 'innovation', 'innovative',
                             'solution', 'solutions', 'potential', 'enable',
                             'enables', 'enabling', 'renewable', 'renewables',
                             'efficiency', 'efficient', 'savings', 'transition'])

    tokens = word_tokenize(doc)
    new_features.append(len([x for x in tokens if x in risk_words]))
    new_features.append(len([x for x in tokens if x in opportunity_words]))

    # Don't modify below here
    return np.hstack([slp_features_for_doc(doc), np.array(new_features)])

In [ ]:
new_train_features = np.vstack([new_features_for_doc(x) for x in train_texts])
# max_iter is the maximum number of iterations during training. It is increased from the 
# default of 100 here to allow training to converge.
# If you encounter a convergence warning you could try increasing this further.
lr_new_classifier = LogisticRegression(random_state=0, max_iter=500).fit(new_train_features, train_labels)
new_dev_features = np.vstack([new_features_for_doc(x) for x in dev_texts])

lr_new_dev_predictions = lr_new_classifier.predict(new_dev_features)
print_results(dev_labels, lr_new_dev_predictions)


In [ ]:
# Here are some example analyses of the behaviour of the logistic regression
# classifier with the example features that could be helpful for identifying 
# interesting features to consider.
#
# Compute a confusion matrix for logistic regression with the example
# features on the dev data. 
# Note that I've transposed the confusion matrix so that 
# the rows correspond to system predictions and columns correspond to
# gold standard labels, following the convention in the textbook 
# and class. (By default, sklearn does it the other way around.)
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html

from sklearn.metrics import confusion_matrix
print(confusion_matrix(dev_labels, lr_slp_dev_predictions, labels=[0, 1, 2]).transpose())

In [ ]:
# Calculte per-class P, R, F for logistic regression with the example
# features on the dev data.
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html

for label in list(range(3)):
    print(label)
    # This prints P, R, F, and support (i.e., number of gold standard instances) for each class
    print(precision_recall_fscore_support(dev_labels, lr_slp_dev_predictions, labels=[label]))

In [ ]:
# Sort the features by their weight for each class for
# the logistic regression classifier using the example features.
# Note that this is a 3-way classification problem. As such, the classifier
# here is multinomial logistic regression, which is slightly different
# than the (binary) logistic regression classifier we saw in lecture. In
# multinomial logistic regression, each class has a separate vector of weights.
# Note that feature numbers below start at 0, i.e., 0 corresponds to x1 in
# the example in the textbook.
for class_label_i,coefs in enumerate(lr_slp_classifier.coef_):
    print("Top features for class", lr_slp_classifier.classes_[class_label_i])
    for fi,val in sorted(enumerate(coefs), key=lambda x : x[1], reverse=True):
        print(fi, val)


## Unigram Count Features

The code below uses sklearn to train a logistic regression classifier with unigram count features on the training data and evaluate it on the dev data. (This document representation is also commonly referred to as a "bag-of-words".)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# sklearn provides functionality for tokenizing text and
# extracting features from it. This uses sklearn's default
# tokenization which applies case folding and treats
# alphanumeric sequences of 2 or more characters as tokens.
# Punctuation is treated as a token separator, i.e.,
# punctuation never occurs in the resulting tokens.
# http://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
# (The word_tokenize function provided above is intended to
# produce the same tokenization.)
count_vectorizer = CountVectorizer()

# train_counts will be a DxV matrix where D is the number of
# training documents and V is the number of word (token) types 
# in the training documents. Each cell in the matrix indicates 
# the frequency (count) of a word type in a document.
train_counts = count_vectorizer.fit_transform(train_texts)

# Train a logistic regression classifier on the training
# data. A wide range of options are available. sklearn uses
# L2 regularization by default. The algorithm for minimizing
# the loss during training is LBFGS instead of SGD, which we
# saw in lecture. The random_state is set to 0  (an 
# arbitrarily chosen number) to help ensure results are 
# consistent from run to run.
# http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
lr = LogisticRegression(random_state=0)
lr_classifier = lr.fit(train_counts, train_labels)

# Transform the dev documents into a DxV matrix, similar to
# that for the training documents, where D is the number of
# dev documents, and V is the number of word types in the 
# training documents.
dev_counts = count_vectorizer.transform(dev_texts)

# Predict the class for each dev document. 
lr_dev_predictions = lr_classifier.predict(dev_counts)

print_results(dev_labels, lr_dev_predictions)

## Part 2: Tf-idf (2 marks)

We will compare logistic regression with unigram count features to the same approach but using tf-idf weighting instead of raw counts. I.e., here we will consider logistic regression with tf-idf weighted unigram features.

Using the code above for unigram count features as an example, and documentation below on the sklearn TfidfVectorizer, add code below to implement this tf-idf approach. As in the above example, your classifier should be trained on the training data and evaluated on the dev data. In implementing this approach, you should aim to make your classifier the same as the above example, but with one difference: the features should use tf-idf weighting instead of simple counts.

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

You should not import any additional modules. 

Hint: This should not be a huge amount of code. (My sample solution is less than 10 lines of code.) If you are writing a lot of code you might be off track.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_train_features = tfidf_vectorizer.fit_transform(train_texts)
lr_tfidf_classifier = LogisticRegression(random_state=0).fit(
    tfidf_train_features, train_labels)
tfidf_dev_features = tfidf_vectorizer.transform(dev_texts)
tfidf_dev_predictions = lr_tfidf_classifier.predict(tfidf_dev_features)
print_results(dev_labels, tfidf_dev_predictions)


## Part 3: Other Features (4 marks)

In this part of the assignment you will again consider logistic regression with features of your choosing. Based on the discussion of document classification, logistic regression, and sentiment analysis in class, or analysis of the training and development data, identify a document representation and corresponding features that you believe could potentially improve over tf-idf weighted unigram features.

(Examining a confusion matrix or the per-class P, R, and F1 values for the classifier using tf-idf weighted unigram features could be helpful. The analysis code from Part 1 provides examples that you could adapt for this. I have included an additional example below for examining feature weights in the classifier using unigram count features that could also be helpful. You are of course welcome to carry out further analyses, although you should not look at the test data yet.)

Implement a logistic regression classifier using these features, train it on the training data, and evaluate it on the dev data.

Here are some ideas for features, but you are by no means limited to them!

* word $n$-grams (perhaps using tf-idf weighting)
* character $n$-grams (again perhaps using tf-idf weighting)
* tf-idf weighted unigram features, but incorporating case folding (or not), stop-word removal, stemming, etc.
* tf-idf weighted unigram features, but using alternative tokenization methods (perhaps incorporating information that isn't present in the default tokenization, such as punctuation)

Importantly, although you need to be able to explain why the features you consider could potentially improve over tf-idf weighted unigram features, it is not a requirement for this assignment that these features actually do improve over tf-idf weighted unigram features. (Many approaches that you might reasonably expect to give an improvement turn out not to for this dataset. For this part of the assignment, the motivation for considering the features and discussion of findings is more important than getting a high F1 or accuracy.)

For implementing your features, note that the constructor for sklearn's TfidfVectorizer has many optional parameters. For example, ```ngram_range``` could be helpful for some of the suggested features above. If you would like to explore alternative approaches to tokenization, ```tokenizer``` allows you to pass your own tokenizer.
https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

For this part of the assignment, you are welcome to import other modules from within sklearn, nltk, numpy, or scipy; however, you must use the sklearn logistic regression classifier.

In the approach you consider, it might be necessary to set some hyper-parameters. For example, if you consider word $n$-gram features, it might be necessary to set the value of $n$. You could do this by grid search over a reasonable range of values, training on the training data and evaluating on the dev data. If you explore the use of normalization approaches, such as case folding and stop-word removal, you could explore combinations of these on the development data to identify the best approach. (Importantly, you should not consider the test data at this point.)


Answer the following questions:

1. Which features are you using? Provide a brief explanation.

I combine word unigram TF--IDF features with character 3--4-gram TF--IDF features. Character features are extracted within word boundaries, terms occurring in fewer than two training documents are omitted, and the character feature block is weighted by 0.5 relative to the word block. I selected these settings using only development-set performance.
   
1. Why might we reasonably expect that a logistic regression classifier using these features would potentially improve over a logistic regression classifier using tf-idf weighted unigram features?

Character n-grams can represent informative word stems and related forms (for example, *risk/risks/risky*), as well as parts of rare or previously unseen words. They can therefore generalize across morphology and spelling variants that separate word unigrams treat as unrelated features. Keeping word unigrams alongside them preserves the strong whole-word evidence, while down-weighting the character block prevents its much larger feature set from dominating.


In [ ]:
from sklearn.pipeline import FeatureUnion

other_vectorizer = FeatureUnion(
    [('word', TfidfVectorizer()),
     ('character', TfidfVectorizer(analyzer='char_wb',
                                   ngram_range=(3, 4), min_df=2,
                                   max_features=30000))],
    transformer_weights={'word': 1.0, 'character': 0.5})
other_train_features = other_vectorizer.fit_transform(train_texts)
lr_other_classifier = LogisticRegression(random_state=0, max_iter=500).fit(
    other_train_features, train_labels)
other_dev_features = other_vectorizer.transform(dev_texts)
other_dev_predictions = lr_other_classifier.predict(other_dev_features)
print_results(dev_labels, other_dev_predictions)

In [ ]:
# Find the top-10 words (features) with highest weight for each class for
# the logistic regression classifier using unigram count features.
# It could be helpful to adapt this to your classifier using
# tf-idf weighted unigram features and your tfidf Vectorizer.
for class_label_i,coefs in enumerate(lr_classifier.coef_):
    print("Top-10 words for class", lr_classifier.classes_[class_label_i])
    top_n = sorted(enumerate(coefs), key=lambda x : x[1], reverse=True)[:10]
    for vocab_i,weight in top_n:
        print (count_vectorizer.get_feature_names_out()[vocab_i], weight)
    print()


## Part 4: Test Data and Report (2 marks)

So far, we've only evaluated on the development data. Once you have completed the tasks above, run the code below to evaluate the classifiers on the test data.

In [ ]:
test_texts,test_labels = get_texts_and_labels('data/test.csv')

print("Test results:")
print()

print("Baseline:")
baseline_test_predictions = [baseline_classifier.predict(x) for x in test_texts]
print_results(test_labels, baseline_test_predictions)

print("SLP Example Features")
slp_test_features = np.vstack([slp_features_for_doc(x) for x in test_texts])
slp_test_predictions = lr_slp_classifier.predict(slp_test_features)
print_results(test_labels, slp_test_predictions)

print("SLP Example + Extended Features")
new_test_features = np.vstack([new_features_for_doc(x) for x in test_texts])
new_test_predictions = lr_new_classifier.predict(new_test_features)
print_results(test_labels, new_test_predictions)

print("Unigram Count Features")
test_counts = count_vectorizer.transform(test_texts)
lr_test_predictions = lr_classifier.predict(test_counts)
print_results(test_labels, lr_test_predictions)

print("Tf-idf Weighted Unigram Features")
tfidf_test_features = tfidf_vectorizer.transform(test_texts)
tfidf_test_predictions = lr_tfidf_classifier.predict(tfidf_test_features)
print_results(test_labels, tfidf_test_predictions)

print("Part 3: Other Features")
other_test_features = other_vectorizer.transform(test_texts)
other_test_predictions = lr_other_classifier.predict(other_test_features)
print_results(test_labels, other_test_predictions)



In this assignment you have considered several logistic regression classifiers. Write a brief report describing your findings that addresses at least the following points. In addressing these points, focus on results on the test data, but you are welcome to also comment on results on dev data where it is relevant to your claims.

1. Do the features you added in Part 1 improve over the example features. I.e., does logistic regression using the example features and the features you added in Part 1 improve over logistic regression using just the example features?

1. Does tf-idf weighting improve performance? I.e., does logistic regression with tf-idf weighted unigram features improve over logistic regression with unigram count features?

1. Does logistic regression using the features of your choice (from Part 3) improve over logistic regression using tf-idf weighted unigram features? Provide some explanation for why this might be the case.

For example, if logistic regression using the features of your choice outperforms logistic regression using tf-idf weighted unigram features, you could carry out some error analysis to attempt to explain what causes the difference. For this, you might find it helpful to examine the per-class P, R, and F values, or a confusion matrix. (Examples of how to do this are given above on the dev data, but here you would want to consider the test data.)

On the other hand, if logistic regression using the features of your choice does not outperform logistic regression using tf-idf weighted unigram features, attempt to explain why this might be the case.


TODO: Write your report here